# 03 — HNSW Benchmarks

We sweep:

* `M` — number of edges per node (memory & build cost driver)
* `efConstruction` — build-time candidate width
* `efSearch` — query-time candidate width (recall ↔ QPS tunable)

Two studies:
1. Vary `M` (efConstruction fixed) → effect of graph degree
2. Vary `efConstruction` (M fixed) → effect of build quality
3. Heatmap `M × efSearch` → Recall@100

Notebook uses the same `data/imagenet1m.h5` + `BASE_PATH` memmap as notebook 02.

In [ ]:
import os, sys, time, gc, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import faiss
import psutil
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd()))
import utils

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

DATA = Path('data')
LAB_LIGHT = int(os.environ.get('LAB_LIGHT', '0'))
OUT_RUN = utils.run_mode()
RESULTS = utils.results_dir()
DOCS_IMG = utils.plots_dir()
print(f'OUT_RUN={OUT_RUN}  RESULTS={RESULTS}  DOCS_IMG={DOCS_IMG}')

# Load query + GT
with h5py.File(DATA / 'imagenet1m.h5', 'r') as h:
    queries = np.array(h['query'], dtype=np.float32)
    gt = np.array(h['groundtruth'], dtype=np.int32)
    DIM = int(h.attrs['dim']); N_BASE = int(h.attrs['n_base'])
    BASE_PATH = str(h.attrs['base_path'])

# Make BASE_PATH portable: prefer the file as it currently exists on disk
# (host path stored in h5 may not match container/CI paths).
_local = DATA / 'imagenet_base.fvecs'
if not Path(BASE_PATH).exists() and _local.exists():
    BASE_PATH = str(_local.resolve())
print('BASE_PATH =', BASE_PATH)

print('queries', queries.shape, 'gt', gt.shape, 'dim', DIM, 'base', N_BASE)
print(f"threads={faiss.omp_get_max_threads()}  RAM free={psutil.virtual_memory().available/1e9:.1f} GB")

In [ ]:
N_SWEEP = int(os.environ.get('LAB_N_SWEEP', 500_000))
LAB_LIGHT = int(os.environ.get('LAB_LIGHT', '0'))

# Study 1: vary M, fixed ef_construction
M_GRID = [8, 16, 32, 48]
EFC_FIXED = 200
EFS_GRID = [10, 20, 40, 80, 160, 320, 640]
# Study 2: vary ef_construction, fixed M
M_FIXED = 32
EFC_GRID = [40, 100, 200, 400]

QUERY_K = 100
if LAB_LIGHT:
    M_GRID = [8, 16, 32]
    EFS_GRID = [10, 20, 40, 80, 160, 320]
    EFC_GRID = [40, 100, 200]
QPS_REPEAT = int(os.environ.get('LAB_QPS_REPEAT', '2' if LAB_LIGHT else '1'))
QPS_WARMUP = int(os.environ.get('LAB_QPS_WARMUP', '1' if LAB_LIGHT else '0'))
_default_qn = queries.shape[0] if LAB_LIGHT else min(10000, queries.shape[0])
QUERY_N = int(os.environ.get('LAB_QUERY_N', str(_default_qn)))
queries_sweep = queries[:QUERY_N]
print(f"N_SWEEP={N_SWEEP:,}  LAB_LIGHT={LAB_LIGHT}  M_GRID={M_GRID}  EFC_FIXED={EFC_FIXED}  EFS_GRID={EFS_GRID}")
print(f"M_FIXED={M_FIXED}  EFC_GRID={EFC_GRID}")
print(f"QPS_REPEAT={QPS_REPEAT}  QPS_WARMUP={QPS_WARMUP}  QUERY_N={QUERY_N}")

In [ ]:
def ensure_gt(n: int, k: int = QUERY_K):
    cache = DATA / f'gt_n{n}_k{k}.npy'
    if cache.exists():
        return np.load(cache)
    print(f'Computing exact GT (Flat) on first {n:,} base vectors × {queries.shape[0]:,} queries × k={k}...')
    flat = faiss.IndexFlatL2(DIM)
    utils.stream_add(flat, BASE_PATH, n)
    _, I = flat.search(queries, k)
    np.save(cache, I)
    del flat; gc.collect()
    return I

gt_local = ensure_gt(N_SWEEP)
utils.print_mem('after GT')
print('gt_local', gt_local.shape, '  (base streamed from disk, not held in RAM)')

## Study 1 — vary M

Each `M` value runs in its own cell (CSV checkpoint).

In [ ]:
HNSW_VARYM_PATH = RESULTS / 'hnsw_varyM.csv'
utils.init_results_csv(HNSW_VARYM_PATH)
print('HNSW vary-M checkpoint:', HNSW_VARYM_PATH)

#### HNSW study 1 — M=8

In [ ]:
if 8 not in M_GRID:
    print('skip HNSW M=8')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, 8, faiss.METRIC_L2)
    idx.hnsw.efConstruction = EFC_FIXED
    with utils.timed('build M=8', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[M={8:3}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=8, efConstruction=EFC_FIXED, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyM'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYM_PATH)

#### HNSW study 1 — M=16

In [ ]:
if 16 not in M_GRID:
    print('skip HNSW M=16')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, 16, faiss.METRIC_L2)
    idx.hnsw.efConstruction = EFC_FIXED
    with utils.timed('build M=16', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[M={16:3}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=16, efConstruction=EFC_FIXED, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyM'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYM_PATH)

#### HNSW study 1 — M=32

In [ ]:
if 32 not in M_GRID:
    print('skip HNSW M=32')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, 32, faiss.METRIC_L2)
    idx.hnsw.efConstruction = EFC_FIXED
    with utils.timed('build M=32', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[M={32:3}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=32, efConstruction=EFC_FIXED, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyM'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYM_PATH)

#### HNSW study 1 — M=48

In [ ]:
if 48 not in M_GRID:
    print('skip HNSW M=48')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, 48, faiss.METRIC_L2)
    idx.hnsw.efConstruction = EFC_FIXED
    with utils.timed('build M=48', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[M={48:3}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=48, efConstruction=EFC_FIXED, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyM'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYM_PATH)

In [ ]:
df_M = pd.read_csv(HNSW_VARYM_PATH)
display(df_M.tail(8))

### Plot 1 — efSearch vs Recall@100 (per M)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for M, sub in df_M.groupby('M'):
    sub = sub.sort_values('efSearch')
    ax.plot(sub.efSearch, sub.recall_100, marker='o', label=f'M={M}')
ax.set_xscale('log'); ax.set_xlabel('efSearch'); ax.set_ylabel('Recall@100')
ax.set_title(f'HNSW — Recall@100 vs efSearch  (efC={EFC_FIXED}, N={N_SWEEP:,})')
ax.legend(); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(DOCS_IMG / '03_hnsw_recall_vs_efs.png', dpi=120); plt.show()

### Plot 2 — QPS vs Recall@100 Pareto (per M)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette('plasma', len(M_GRID))
for col, (M, sub) in zip(palette, df_M.groupby('M')):
    ax.scatter(sub.recall_100, sub.qps, color=col, label=f'M={M}', s=50, edgecolors='k')
    ms = utils.pareto_frontier(sub.recall_100.values, sub.qps.values)
    o = np.argsort(sub.recall_100.values[ms])
    ax.plot(sub.recall_100.values[ms][o], sub.qps.values[ms][o], color=col, lw=0.8, ls='--', alpha=0.85)
    pidx = np.where(ms)[0]
    if len(pidx) <= 6:
        for i in pidx:
            r = sub.iloc[i]
            ax.annotate(f"efS={int(r.efSearch)}", (r.recall_100, r.qps), fontsize=6, alpha=0.75,
                        xytext=(3, 3), textcoords='offset points')
ax.set_yscale('log'); ax.set_xlabel('Recall@100'); ax.set_ylabel('QPS (log)')
ax.set_title('HNSW — Pareto QPS vs Recall@100  (efC={}; labels on Pareto only)'.format(EFC_FIXED))
ax.legend()
plt.tight_layout(); plt.savefig(DOCS_IMG / '03_hnsw_pareto_M.png', dpi=120); plt.show()

### Plot 3 & 4 — Build time / index size vs M

In [ ]:
df_bt = df_M.drop_duplicates('M')[['M', 'build_s', 'size_mb']]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=df_bt, x='M', y='build_s', ax=ax[0], color='steelblue')
ax[0].set_title('HNSW build time'); ax[0].set_ylabel('seconds')
sns.barplot(data=df_bt, x='M', y='size_mb', ax=ax[1], color='darkorange')
ax[1].set_title('HNSW index size'); ax[1].set_ylabel('MB')
plt.tight_layout(); plt.savefig(DOCS_IMG / '03_hnsw_build_size_M.png', dpi=120); plt.show()
display(df_bt.reset_index(drop=True))

## Study 2 — vary efConstruction (M=32)

Each `efConstruction` value runs in its own cell (CSV checkpoint).

In [ ]:
HNSW_VARYEFC_PATH = RESULTS / 'hnsw_varyEFC.csv'
utils.init_results_csv(HNSW_VARYEFC_PATH)
print('HNSW vary-efC checkpoint:', HNSW_VARYEFC_PATH)

#### HNSW study 2 — efConstruction=40

In [ ]:
if 40 not in EFC_GRID:
    print('skip HNSW efC=40')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, M_FIXED, faiss.METRIC_L2)
    idx.hnsw.efConstruction = 40
    with utils.timed('build efC=40', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[efC={40:4}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=M_FIXED, efConstruction=40, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyEFC'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYEFC_PATH)

#### HNSW study 2 — efConstruction=100

In [ ]:
if 100 not in EFC_GRID:
    print('skip HNSW efC=100')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, M_FIXED, faiss.METRIC_L2)
    idx.hnsw.efConstruction = 100
    with utils.timed('build efC=100', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[efC={100:4}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=M_FIXED, efConstruction=100, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyEFC'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYEFC_PATH)

#### HNSW study 2 — efConstruction=200

In [ ]:
if 200 not in EFC_GRID:
    print('skip HNSW efC=200')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, M_FIXED, faiss.METRIC_L2)
    idx.hnsw.efConstruction = 200
    with utils.timed('build efC=200', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[efC={200:4}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=M_FIXED, efConstruction=200, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyEFC'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYEFC_PATH)

#### HNSW study 2 — efConstruction=400

In [ ]:
if 400 not in EFC_GRID:
    print('skip HNSW efC=400')
else:
    rows = []
    idx = faiss.IndexHNSWFlat(DIM, M_FIXED, faiss.METRIC_L2)
    idx.hnsw.efConstruction = 400
    with utils.timed('build efC=400', sample_rss_peak=True) as tb:
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[efC={400:4}]  build {tb.elapsed:7.1f}s · size {size_mb:7.1f} MB')
    for efs in EFS_GRID:
        idx.hnsw.efSearch = efs
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='HNSW', M=M_FIXED, efConstruction=400, efSearch=efs,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP, study='varyEFC'))
        print(f'    efS={efs:4}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx; gc.collect()
    utils.append_results(rows, HNSW_VARYEFC_PATH)

In [ ]:
df_EFC = pd.read_csv(HNSW_VARYEFC_PATH)
display(df_EFC.tail(8))

### Plot 5 — Recall@100 vs efSearch (per efConstruction, M fixed)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for efc, sub in df_EFC.groupby('efConstruction'):
    sub = sub.sort_values('efSearch')
    ax.plot(sub.efSearch, sub.recall_100, marker='o', label=f'efC={efc}')
ax.set_xscale('log'); ax.set_xlabel('efSearch'); ax.set_ylabel('Recall@100')
ax.set_title(f'HNSW — efSearch vs Recall@100  (M={M_FIXED})')
ax.legend(); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(DOCS_IMG / '03_hnsw_recall_vs_efs_EFC.png', dpi=120); plt.show()

### Plot 6 — Build time vs efConstruction

In [ ]:
df_bt2 = df_EFC.drop_duplicates('efConstruction')[['efConstruction', 'build_s']]
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=df_bt2, x='efConstruction', y='build_s', ax=ax, color='seagreen')
ax.set_title(f'HNSW build time vs efConstruction (M={M_FIXED})')
ax.set_ylabel('seconds')
plt.tight_layout(); plt.savefig(DOCS_IMG / '03_hnsw_build_vs_EFC.png', dpi=120); plt.show()
display(df_bt2.reset_index(drop=True))

### Plot 7 — Heatmap M × efSearch → Recall@100

In [ ]:
piv = df_M.pivot(index='M', columns='efSearch', values='recall_100')
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(piv, annot=True, fmt='.3f', cmap='viridis', vmin=0, vmax=1, ax=ax)
ax.set_title(f'HNSW — Recall@100 heatmap  (efC={EFC_FIXED}, N={N_SWEEP:,})')
plt.tight_layout(); plt.savefig(DOCS_IMG / '03_hnsw_heatmap.png', dpi=120); plt.show()

# Combined CSV for notebook 05
df_all_hnsw = pd.concat([df_M, df_EFC], ignore_index=True)
df_all_hnsw.to_csv(RESULTS / 'hnsw_all.csv', index=False)
print('rows', len(df_all_hnsw))

## Summary

* Larger `M` → higher recall and bigger index, with diminishing return after M ≈ 32.
* Higher `efConstruction` mostly helps **at the top of the recall range**
  (`Recall@100 ≥ 0.9`).
* `efSearch` is a *runtime knob* — adjustable without rebuilding.

See `05_comparison.ipynb` for cross-algorithm comparison.